In [2]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 5 modules, rebound 1 names.


In [3]:
"""Step 0 - reference harness.
 
Run once with the official `tirex2` package installed. It
  1. prints the checkpoint config (the real hyperparameters you will mirror),
  2. prints every state_dict key + shape (your modules must reproduce these names),
  3. runs a fixed test batch through the official model and stores every intermediate
     activation in `ref_acts.pt`, so each later step can be checked in isolation.
 
Usage:  python step0_reference.py --device cuda
"""

'Step 0 - reference harness.\n\nRun once with the official `tirex2` package installed. It\n  1. prints the checkpoint config (the real hyperparameters you will mirror),\n  2. prints every state_dict key + shape (your modules must reproduce these names),\n  3. runs a fixed test batch through the official model and stores every intermediate\n     activation in `ref_acts.pt`, so each later step can be checked in isolation.\n\nUsage:  python step0_reference.py --device cuda\n'

In [4]:
import argparse
 
import torch
import yaml
from huggingface_hub import hf_hub_download
 
from tirex2 import TimeseriesType, load_model

In [5]:
PRED_LEN = 64

In [6]:
def make_test_series(seed: int = 0) -> list[TimeseriesType]:
    """Small batch that hits every code path: truncation, left padding, NaNs, binary rows,
    multiple targets, past-only covariates and future-known covariates."""
    g = torch.Generator().manual_seed(seed)
    t = torch.arange(3000, dtype=torch.float32)
 
    # 1) univariate, longer than context_len -> gets truncated
    s1 = TimeseriesType(
        target=(10 + torch.sin(t / 20) + 0.1 * torch.randn(3000, generator=g))[None],
        past_covariates=None,
        future_covariates=None,
    )
 
    # 2) univariate, short with missing values -> left NaN padding + NaN mask channel
    y2 = torch.cos(t[:300] / 7) * 50 + 100
    y2[40:60] = torch.nan
    s2 = TimeseriesType(target=y2[None], past_covariates=None, future_covariates=None)
 
    # 3) 2 targets + 1 past-only covariate + 2 future-known covariates (one of them binary)
    T3 = 1000
    x = t[: T3 + PRED_LEN]
    driver = torch.sin(x / 15)                           # known in the future
    holiday = ((x.long() % 50) < 3).float()              # binary, known in the future
    tgt = torch.stack([3 * driver[:T3] + 5 * holiday[:T3], 0.01 * x[:T3] + driver[:T3]])
    tgt = tgt + 0.05 * torch.randn(tgt.shape, generator=g)
    past_cov = torch.randn(1, T3, generator=g).cumsum(-1)  # past-only
    s3 = TimeseriesType(
        target=tgt,
        past_covariates=past_cov,
        future_covariates=torch.stack([driver, holiday]),
    )
    return [s1, s2, s3]
 
 
@torch.no_grad()
def capture(ref, series, prediction_length: int = PRED_LEN) -> dict:
    """Collect activations at every interface of the official model (no TTA, no differencing)."""
    acts: dict = {}
 
    # time-mixer outputs (inside each block, before the variate mixer)
    hooks = [
        blk.time_mixer.register_forward_hook(lambda m, inp, out, i=i: acts.__setitem__(f"time_mixer_{i}", out))
        for i, blk in enumerate(ref.stack)
    ]
 
    # tta_diff=False: keep the postprocessor out of the comparison for now
    _, batch, _, post_kwargs = ref._prepare_input(series, prediction_length, False)
    x, group_vector, target_mask, known_cov_mask, scaler_state, tok_state = ref._prepare_forward(batch)
 
    acts["batch"] = batch                      # x [BV, T], group_vector [BV], target_mask [BV]
    acts["known_covariate_mask"] = known_cov_mask
    acts["scaler_state"] = scaler_state        # (loc, scale, is_binary)
    acts["embed"] = x                          # [BV, L, D]
    acts["blocks"] = list(ref._forward_layerwise(x, group_vector, target_mask, known_cov_mask))
    acts["model_out"] = ref(batch)             # [BV, Q, L*P] in data scale
    acts["forecast"] = ref.predict(series, prediction_length, tta_diff=False, tta_sign_flip=False)
 
    for h in hooks:
        h.remove()
    return acts
 
 
def to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: to_cpu(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(to_cpu(v) for v in obj)
    return obj


def check(name, a, b, atol=1e-5, rtol=1e-5):
    ok = torch.allclose(a, b, atol=atol, rtol=rtol, equal_nan=True)
    err = (a - b).nan_to_num().abs().max().item()
    print(f"[{'OK ' if ok else 'FAIL'}] {name:30s} max|diff|={err:.2e}")
    assert ok, name

In [ ]:
REPO = "NX-AI/TiRex-2"
device = "cuda"

cfg = yaml.safe_load(open(hf_hub_download(REPO, "model-config.yaml")))
print("=" * 30, "model-config.yaml", "=" * 30)
print(yaml.safe_dump(cfg, sort_keys=False))

ref = load_model(REPO, device=device).model  # unwrap ForecastModel -> TiRex2 nn.Module

print("=" * 30, "state_dict", "=" * 30)
for k, v in ref.state_dict().items():
    print(f"{k:90s} {tuple(v.shape)}")

series = make_test_series()
acts = capture(ref, series)
torch.save({"series": series, "acts": to_cpu(acts), "config": cfg}, "ref_acts.pt")
print("saved ref_acts.pt | BV =", acts["batch"]["x"].shape[0], "| L =", acts["embed"].shape[1])

============================== model-config.yaml ==============================
num_blocks: 12
embedding_dim: 512
input_patch_size: 32
output_patch_size: 32
input_ff_dim: 2048
h_expand: 4
context_len: 2048
future_len: 320
dropout: 0.1
quantiles:
- 0.1
- 0.2
- 0.3
- 0.4
- 0.5
- 0.6
- 0.7
- 0.8
- 0.9
act_func: torch.nn.SiLU
use_qk_norm: true
tokenizer_cfg:
  input_patch_size: 32
  output_patch_size: 32
  input_patch_stride: 32
scaler_cfg:
  use_arcsinh: true
  binaryaware: true
stack_out_norm_config:
  eps: 1.0e-07
tta_diff: true
stack_config:
  templates:
    slstm_attn:
      time_mixer:
        model_type: bi-slstm
        embedding_dim: 512
        num_heads: 4
        rnn_backend: flashrnn-cuda
        num_slstm_heads: 4
      variate_mixer:
        embedding_dim: 512
        num_heads: 4
      dropout: 0.1
      eps: 1.0e-07
    mlstm_attn:
      time_mixer:
        model_type: bi-mlstm
        embedding_dim: 512
        num_heads: 4
        rnn_backend: flashrnn-cuda
      variate

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

============================== state_dict ==============================
input_patch_embedding.hidden_layer.weight                                                  (2048, 64)
input_patch_embedding.hidden_layer.bias                                                    (2048,)
input_patch_embedding.output_layer.weight                                                  (512, 2048)
input_patch_embedding.output_layer.bias                                                    (512,)
input_patch_embedding.residual_layer.weight                                                (512, 64)
input_patch_embedding.residual_layer.bias                                                  (512,)
stack_out_norm.weight                                                                      (512,)
stack.0.time_mixer.norm_lstm.weight                                                        (512,)
stack.0.time_mixer.fwd_cell.q.weight                                                       (256, 512)
stack.0.time_mixer.fwd_cell.

AttributeError: 'TiRex2' object has no attribute '_prepare_input'

# Scaler

In [7]:
from tirex2.model.component.scaler import Scaler as RefScaler

from ts_uncertainty.models.tirex2.scaling import Scaler as MyScaler

In [11]:
# sample data to test the scaler
torch.manual_seed(0)
x = torch.randn(6, 2368) * 5 + 3
x[0, :500] = torch.nan                          # left padding
x[1] = (torch.rand(2368) > 0.5).float()         # binary row
x[1, -320:] = torch.nan
x[2] = torch.nan                                # all-NaN row
x[3] = 7.0                                      # constant row -> scale clamps to eps
x[4, -320:] = torch.nan                         # unknown future
# x = x.unsqueeze(0) # add batch dimension, [B, V, T]

In [12]:
ref, mine = RefScaler(use_arcsinh=True, binaryaware=True), MyScaler()

z_ref, s_ref = ref.scale(x)
z_my, s_my = mine.scale(x.unsqueeze(0))  # add batch dimension for mine

check("scale", z_ref, z_my.squeeze(0))
y = torch.randn(6, 9, 2368)
y_ref = ref.re_scale(y, s_ref)
y_my = mine.re_scale(y.unsqueeze(0), s_my)
check("re_scale", y_ref, y_my.squeeze(0))


[OK ] scale                          max|diff|=0.00e+00
[OK ] re_scale                       max|diff|=0.00e+00


# Patching

In [28]:
from tirex2.model.component.patch_tokenizer import Tokenizer as RefTokenizer

from ts_uncertainty.models.tirex2.patching import build_context_patches as MyPatcher

In [48]:
patch_size = 16
context_length = 2368

ref = RefTokenizer(patch_size, patch_size, patch_size)

ref_feat, _ = ref.input_transform(x)
my_feat, my_mask = MyPatcher(x.unsqueeze(0), patch_size, context_length)

print("ref_feat.shape", ref_feat.shape, "\n", 
      "my_feat.shape", my_feat.shape, "\n", 
      "my_mask.shape", my_mask.shape)

ref_feat.shape torch.Size([6, 148, 16]) 
 my_feat.shape torch.Size([1, 6, 148, 32]) 
 my_mask.shape torch.Size([1, 6, 148, 16])


# Embedding

In [53]:
from ts_uncertainty.models.tirex2.residual import ResidualBlock

In [54]:
# embedding check
embbedding_size = 64
embed = ResidualBlock(in_dim=2*patch_size, h_dim=128, out_dim=embbedding_size)
ctx_tok = embed(my_feat)
B, V, N, _ = my_feat.shape

print("tokens:", ctx_tok.shape)
print("any NaN in tokens:", torch.isnan(ctx_tok).any().item())

tokens: torch.Size([1, 6, 148, 64])
any NaN in tokens: False
